# EAD macro sensitivity test

A loan-only prepayment model is compared with a loan-plus-macro model on an out-of-time sample.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

In [2]:
ead_macro = query('select * from ead_macro_sensitivity'); ead_macro

,model,sample,rows,events,roc_auc,macro_auc_improvement,production_conclusion
0,loan_only,out_of_time,80872,545,0.3998,-0.0007,retain scenario-invariant EAD
1,loan_plus_macro,out_of_time,80872,545,0.3991,-0.0007,retain scenario-invariant EAD


In [3]:
query('''
select future_month, max(ead)-min(ead) scenario_ead_difference
from ead_term_structure
where stage=2
group by future_month
order by future_month
limit 24
''')

,future_month,scenario_ead_difference
0,1,0.0000
1,2,0.0000
2,3,0.0000
3,4,0.0000
4,5,0.0000
5,6,0.0000
6,7,0.0000
7,8,0.0000
8,9,0.0000
9,10,0.0000


This test concerns voluntary-prepayment ranking. It does not directly model the balance conditional on default. If the macro variables do not add material ranking power, the production EAD remains scenario-invariant and scenario sensitivity enters PD and LGD instead. The displayed scenario mortgage-rate path does not override contractual EIR discounting or conditional EAD.